In [4]:
import pandas as pd

df = pd.read_csv('/content/train[1].csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
# Basic shape and structure
print("Shape:", df.shape)
print("\nColumn Data Types:")
print(df.dtypes)

# Missing values per column
print("\nMissing Values:")
print(df.isnull().sum())

# Percentage of missing values (more useful for reporting)
print("\nMissing Value Percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

# Duplicate rows
print("\nDuplicate Rows:", df.duplicated().sum())

# Quick look at value ranges for numeric columns (helps spot anomalies)
df.describe()

Shape: (891, 12)

Column Data Types:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object
dtype: object

Missing Values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Missing Value Percentage:
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64

Duplicate Rows: 0


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
# 1. Age: 19.87% missing — fill with median (robust to outliers, unlike mean)
df['Age'] = df['Age'].fillna(df['Age'].median())

# 2. Embarked: only 0.22% missing — fill with the most common value (mode)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# 3. Cabin: 77% missing — too much to impute meaningfully, so we drop the column entirely
df = df.drop(columns=['Cabin'])

# Confirm no more missing values (except Cabin, which we dropped)
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [7]:
# Duplicate removal (already confirmed 0 earlier, but we do this step formally)
before_dupes = df.duplicated().sum()
df = df.drop_duplicates()
after_dupes = df.duplicated().sum()
print(f"Duplicates before: {before_dupes}, Duplicates after: {after_dupes}")

# Standardization: check unique values in categorical columns for inconsistent formatting
print("\nUnique values in 'Sex':", df['Sex'].unique())
print("Unique values in 'Embarked':", df['Embarked'].unique())

# Data type correction: Pclass and Survived are categories, not true numbers — but we'll leave
# them as int since they're already clean and usable. PassengerId is fine as int too.

# Outlier detection using IQR method on Fare (Age is now cleaned with median, less relevant for outliers)
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Fare'] < lower_bound) | (df['Fare'] > upper_bound)]
print(f"\nNumber of Fare outliers: {len(outliers)}")
print(f"Outlier bounds: below {lower_bound:.2f} or above {upper_bound:.2f}")

Duplicates before: 0, Duplicates after: 0

Unique values in 'Sex': ['male' 'female']
Unique values in 'Embarked': ['S' 'C' 'Q']

Number of Fare outliers: 116
Outlier bounds: below -26.72 or above 65.63


In [8]:
# Data type correction: ensure PassengerId is treated as an identifier (string), not a number to be summed/averaged
df['PassengerId'] = df['PassengerId'].astype(str)

# Before vs After summary table
summary = pd.DataFrame({
    'Metric': ['Null Count (Age)', 'Null Count (Cabin)', 'Null Count (Embarked)', 'Duplicate Rows', 'Total Rows', 'Total Columns'],
    'Before Cleaning': [177, 687, 2, 0, 891, 12],
    'After Cleaning': [df['Age'].isnull().sum(), 'Column Dropped', df['Embarked'].isnull().sum(), df.duplicated().sum(), df.shape[0], df.shape[1]]
})
print(summary)

# Save the cleaned dataset
df.to_csv('titanic_cleaned.csv', index=False)
print("\nCleaned dataset saved as 'titanic_cleaned.csv'")

                  Metric  Before Cleaning  After Cleaning
0       Null Count (Age)              177               0
1     Null Count (Cabin)              687  Column Dropped
2  Null Count (Embarked)                2               0
3         Duplicate Rows                0               0
4             Total Rows              891             891
5          Total Columns               12              11

Cleaned dataset saved as 'titanic_cleaned.csv'


## Conclusion

This project demonstrated a complete data cleaning pipeline on the Titanic
dataset, transforming it from a raw state with significant missing data
into an analysis-ready dataset.

**Summary of actions taken:**
- Filled missing Age values (19.87%) using the median
- Filled missing Embarked values (0.22%) using the mode
- Dropped the Cabin column entirely due to 77.10% missing data
- Confirmed 0 duplicate rows
- Verified categorical columns (Sex, Embarked) were already consistently formatted
- Identified 116 Fare outliers using the IQR method and retained them, since
  they represent legitimate high-fare passengers rather than data errors
- Corrected PassengerId to a string type, since it's an identifier, not a
  quantity to be averaged
- Saved the final cleaned dataset as `titanic_cleaned.csv`

The dataset is now free of missing values (except the intentionally dropped
Cabin column) and ready for further analysis or modeling.